In [1]:
from __future__ import annotations

import math
from dataclasses import asdict, dataclass
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def veilige_del(a: float, b: float, fallback: float = float("nan")) -> float:
    return fallback if abs(b) < 1e-16 else a / b


def toon_dataframe(naam: str, df: pd.DataFrame, max_rijen: int = 20) -> None:
    print(f"\n=== {naam.upper()} ===")
    if len(df) > max_rijen:
        print(df.head(max_rijen).to_string(index=False))
        print(f"... ({len(df)} rijen in totaal)")
    else:
        print(df.to_string(index=False))

In [3]:
try:
    resultaten_interface
except NameError:
    resultaten_interface = {
        "export_voor_volgende_modules": {
            "massa_effectief_op_slede_kg":    120.2 , # Tom v19,
            "massa_te_versnellen_door_m1_kg": 120.2 , # Tom v19,
            "Fz_max_N":  1179.3,  # Tom v19
            "Fx_max_N":   105.30377822832081,
            "Mx_max_Nm":   40.95,  # Tom v19
            "My_max_Nm":  527.0,  # Tom v19
        }
    }

try:
    resultaten_spindel
except NameError:
    resultaten_spindel = {
        "export_voor_volgende_modules": {
            "n_bedrijf_rpm":               840.0,
            "F_piek_N":                   1832.0,
            "F_eq_N":                     1707.0,
            "v_piek_m_s":                    0.28,
            "t_versnellen_s":                0.35,
            "t_constante_snelheid_s":        7.507,
            "t_vertragen_s":                 0.35,
            "spoed_spindel_m":               0.020,
            "rendement_spindel":             0.92,
            "nominale_diameter_spindel_m":   0.040,
            "totale_lengte_spindel_m":       2.478,
            "extra_axiale_kracht_N":      1291.0,
            "m1_slag_m":                     2.200,
            "m1_vrije_lengte_m":             2.350,
            "m1_schacht_totaal_m":           2.478,
        }
    }

In [4]:
def _doel_levensduur_uren(
    n_rpm:    float,
    spoed_m:  float,
    doel_km:  float,
) -> float:
    """
    Rekent kilometerdoelstelling om naar uren looptijd.
    L10h = doel_km × 1000 / (n_rpm × spoed_m × 60)
    """
    if n_rpm <= 0.0 or spoed_m <= 0.0:
        return float("nan")
    return doel_km * 1000.0 / (n_rpm * spoed_m * 60.0)

In [5]:
@dataclass(frozen=True)
class ConfiguratieVastLager:
    """
    HIWIN WBK30DF — duplex hoekcontactlager-eenheid, vast-einde.

    Belastingscoëfficiënten voor 7206 BE (α=25°), DF-arrangement:
        P = X×Fr + Y×Fa   als Fa/Fr > e
        P = Fr             als Fa/Fr ≤ e
    """
    naam:        str   = "HIWIN WBK30DF (2× 7206 BE, DF)"

    C_dyn_N:     float = 24_570.0   # N  — paarwaarde, geschat
    C0_N:        float = 26_400.0   # N  — paarwaarde, geschat
    n_max_rpm:   float = 12_000.0   # rpm

    e_coeff:     float = 0.68
    X_hoog:      float = 0.44
    Y_hoog:      float = 0.92
    X_laag:      float = 1.00
    Y_laag:      float = 0.00

    # Minimumlast: P_min ≥ F_min_ratio × Cdyn om roloraan/slippen te voorkomen
    # Ref: SKF General Catalogue, §2.5 / Palmgren (1959)
    F_min_ratio: float = 0.02

    p_exponent:  float = 3.0     # kogellagerwet L10 ∝ (C/P)^3


@dataclass(frozen=True)
class ConfiguratieZwevendLager:
    """HIWIN BK30 — diepgroefkogellager 6206, zwevend-einde (puur radiaal)."""
    naam:        str   = "HIWIN BK30 (6206)"
    C_dyn_N:     float = 19_500.0
    C0_N:        float = 11_200.0
    n_max_rpm:   float = 15_000.0
    F_min_ratio: float = 0.02
    p_exponent:  float = 3.0


@dataclass(frozen=True)
class ConfiguratieLagers:
    """Systeemparameters voor de lagerberekening."""

    lagerspanning_m:      float = 2.478   

    s0_min:               float = 3.0


    doel_levensduur_km:   float = 5_800.0  

    factor_fw:            float = 1.3

In [6]:
def bepaal_lagerkrachten(
    interface_exp: Dict[str, float],
    spindel_exp:   Dict[str, float],
    cfg:           ConfiguratieLagers,
) -> Dict[str, float]:
    """
    Radiale en axiale krachten op vast- en zwevend-einde.

    Axiale kracht:
        Het vastlager neemt de volledige axiale last op (vast-zwevend opstelling).
        F_axiaal = F_piek (uit nb03, incl. gewicht + inertia + fw + wrijving)

    Radiale krachten:
        Gewichtsreactie + buigmomentreactie (My_max) verdeeld over lagerspanning L.
        Vast-einde  : R_vast   = ½ Fz + |My|/L  (ongunstige superpositie)
        Zwevend     : R_zwev   = ½ Fz − |My|/L  (gunstige kant, min 0)

    Noot: Momentbijdrage van Mx (rolverdraaiing) is meegenomen via de
    My-term; Mx << My zodat dit conservatief is.
    """
    Fz   = float(interface_exp.get("Fz_max_N",   1179.3))  # Tom v19
    My   = abs(float(interface_exp.get("My_max_Nm",  527.0)))  # Tom v19
    F_ax = float(spindel_exp.get("F_piek_N",    1832.0))
    L    = cfg.lagerspanning_m

    R_gewicht = Fz / 2.0
    R_moment  = veilige_del(My, L, 0.0)

    F_rad_vast    = R_gewicht + R_moment
    F_rad_zwevend = max(R_gewicht - R_moment, 0.0)

    return {
        "Fz_N":            Fz,
        "My_Nm":           My,
        "F_axiaal_N":      F_ax,
        "R_gewicht_N":     R_gewicht,
        "R_moment_N":      R_moment,
        "F_rad_vast_N":    F_rad_vast,
        "F_rad_zwevend_N": F_rad_zwevend,
    }

In [7]:
def equiv_belasting_hoekcontact(
    Fr: float,
    Fa: float,
    cfg: ConfiguratieVastLager,
) -> float:
    """
    ISO 281 §5.2 equivalente dynamische belasting voor hoekcontactlager.
    P = max(X×Fr + Y×Fa,  P_min)
    """
    if Fr > 0.0:
        ratio = veilige_del(Fa, Fr, 0.0)
        P = (cfg.X_hoog * Fr + cfg.Y_hoog * Fa) if ratio > cfg.e_coeff \
            else (cfg.X_laag * Fr + cfg.Y_laag * Fa)
    else:
        P = cfg.Y_hoog * Fa   # puur axiaal
    P_min = cfg.F_min_ratio * cfg.C_dyn_N
    return max(P, P_min)


def equiv_belasting_diepgroef(Fr: float, cfg: ConfiguratieZwevendLager) -> float:
    """Diepgroefkogellager, puur radiaal — P = max(Fr, P_min)."""
    P_min = cfg.F_min_ratio * cfg.C_dyn_N
    return max(Fr, P_min)

In [8]:
def L10h_iso(C_dyn: float, P_eq: float, p: float, n_rpm: float, fw: float) -> float:
    """
    ISO 281 basislevensduur L10h [uren].
    L10h = (C_dyn / (fw × P_eq))^p × 10⁶ / (60 × n)
    fw = belastingsfactor (schok, wrijving, dynamische effecten)
    """
    if P_eq <= 0.0 or n_rpm <= 0.0:
        return float("inf")
    P_eff = P_eq * fw
    return (C_dyn / P_eff) ** p * 1e6 / (60.0 * n_rpm)


def L10km_uit_uren(L10h: float, n_rpm: float, spoed_m: float) -> float:
    """Omrekening L10h → km spindel-verplaatsing."""
    return L10h * n_rpm * 60.0 * spoed_m / 1000.0

In [9]:
def verifieer_lagers(
    lagerkrachten: Dict[str, float],
    cfg:           ConfiguratieLagers,
    cfg_vast:      ConfiguratieVastLager,
    cfg_zwevend:   ConfiguratieZwevendLager,
    spindel_exp:   Dict[str, float],
) -> Dict:
    n_rpm   = float(spindel_exp.get("n_bedrijf_rpm",   840.0))
    spoed_m = float(spindel_exp.get("spoed_spindel_m", 0.020))

    # Doel levensduur in uren (exact berekend, geen dead code)
    doel_h = _doel_levensduur_uren(n_rpm, spoed_m, cfg.doel_levensduur_km)

    Fr_vast    = lagerkrachten["F_rad_vast_N"]
    Fa_vast    = lagerkrachten["F_axiaal_N"]
    Fr_zwevend = lagerkrachten["F_rad_zwevend_N"]

    P_vast    = equiv_belasting_hoekcontact(Fr_vast, Fa_vast, cfg_vast)
    P_zwevend = equiv_belasting_diepgroef(Fr_zwevend, cfg_zwevend)

    L10h_vast    = L10h_iso(cfg_vast.C_dyn_N,    P_vast,    cfg_vast.p_exponent,    n_rpm, cfg.factor_fw)
    L10h_zwevend = L10h_iso(cfg_zwevend.C_dyn_N, P_zwevend, cfg_zwevend.p_exponent, n_rpm, cfg.factor_fw)

    L10km_vast    = L10km_uit_uren(L10h_vast,    n_rpm, spoed_m)
    L10km_zwevend = L10km_uit_uren(L10h_zwevend, n_rpm, spoed_m)

    # Statische veiligheid: s0 = C0 / P_statisch (ISO 76)
    s0_vast    = veilige_del(cfg_vast.C0_N,    P_vast)
    s0_zwevend = veilige_del(cfg_zwevend.C0_N, P_zwevend)

    benutting_n_vast    = veilige_del(n_rpm, cfg_vast.n_max_rpm)
    benutting_n_zwevend = veilige_del(n_rpm, cfg_zwevend.n_max_rpm)

    resultaten = {
        "doel_levensduur_h":   doel_h,
        "P_eq_vast_N":         P_vast,
        "P_eq_zwevend_N":      P_zwevend,
        "L10h_vast":           L10h_vast,
        "L10h_zwevend":        L10h_zwevend,
        "L10km_vast":          L10km_vast,
        "L10km_zwevend":       L10km_zwevend,
        "s0_vast":             s0_vast,
        "s0_zwevend":          s0_zwevend,
        "benutting_n_vast":    benutting_n_vast,
        "benutting_n_zwevend": benutting_n_zwevend,
        "marge_L10h_vast":     veilige_del(L10h_vast,    doel_h),
        "marge_L10h_zwevend":  veilige_del(L10h_zwevend, doel_h),
    }

    controles = pd.DataFrame([
        {
            "controle": "L10h vast ≥ doel",
            "waarde": L10h_vast, "grens": doel_h,
            "vergelijking": "≥", "eenheid": "h",
            "geslaagd": bool(L10h_vast >= doel_h),
        },
        {
            "controle": "L10h zwevend ≥ doel",
            "waarde": L10h_zwevend, "grens": doel_h,
            "vergelijking": "≥", "eenheid": "h",
            "geslaagd": bool(L10h_zwevend >= doel_h),
        },
        {
            "controle": "L10km vast ≥ doel",
            "waarde": L10km_vast, "grens": cfg.doel_levensduur_km,
            "vergelijking": "≥", "eenheid": "km",
            "geslaagd": bool(L10km_vast >= cfg.doel_levensduur_km),
        },
        {
            "controle": "L10km zwevend ≥ doel",
            "waarde": L10km_zwevend, "grens": cfg.doel_levensduur_km,
            "vergelijking": "≥", "eenheid": "km",
            "geslaagd": bool(L10km_zwevend >= cfg.doel_levensduur_km),
        },
        {
            "controle": "s0 vast ≥ minimum (ISO 76)",
            "waarde": s0_vast, "grens": cfg.s0_min,
            "vergelijking": "≥", "eenheid": "-",
            "geslaagd": bool(s0_vast >= cfg.s0_min),
        },
        {
            "controle": "s0 zwevend ≥ minimum (ISO 76)",
            "waarde": s0_zwevend, "grens": cfg.s0_min,
            "vergelijking": "≥", "eenheid": "-",
            "geslaagd": bool(s0_zwevend >= cfg.s0_min),
        },
        {
            "controle": "n_bedrijf ≤ n_max vast",
            "waarde": n_rpm, "grens": cfg_vast.n_max_rpm,
            "vergelijking": "≤", "eenheid": "rpm",
            "geslaagd": bool(n_rpm <= cfg_vast.n_max_rpm),
        },
        {
            "controle": "n_bedrijf ≤ n_max zwevend",
            "waarde": n_rpm, "grens": cfg_zwevend.n_max_rpm,
            "vergelijking": "≤", "eenheid": "rpm",
            "geslaagd": bool(n_rpm <= cfg_zwevend.n_max_rpm),
        },
    ])

    return {"resultaten": resultaten, "controles": controles, "doel_h": doel_h}

In [10]:
def maak_overzicht_lagers(
    lagerkrachten: Dict,
    controle: Dict,
    cfg_vast: ConfiguratieVastLager,
    cfg_zwevend: ConfiguratieZwevendLager,
) -> pd.DataFrame:
    r = controle["resultaten"]
    return pd.DataFrame([
        ["vast lager",             cfg_vast.naam,         "-"],
        ["zwevend lager",          cfg_zwevend.naam,      "-"],
        ["F_axiaal (vastlager)",   lagerkrachten["F_axiaal_N"],      "N"],
        ["F_radiaal vast",         lagerkrachten["F_rad_vast_N"],    "N"],
        ["F_radiaal zwevend",      lagerkrachten["F_rad_zwevend_N"], "N"],
        ["P_eq vast",              r["P_eq_vast_N"],      "N"],
        ["P_eq zwevend",           r["P_eq_zwevend_N"],   "N"],
        ["Cdyn vast",              cfg_vast.C_dyn_N,      "N"],
        ["Cdyn zwevend",           cfg_zwevend.C_dyn_N,   "N"],
        ["L10h vast",              r["L10h_vast"],        "h"],
        ["L10h zwevend",           r["L10h_zwevend"],     "h"],
        ["Doel L10h",              r["doel_levensduur_h"],"h"],
        ["Marge L10h vast",        r["marge_L10h_vast"],  "-"],
        ["L10km vast",             r["L10km_vast"],       "km"],
        ["L10km zwevend",          r["L10km_zwevend"],    "km"],
        ["s0 vast (ISO 76)",       r["s0_vast"],          "-"],
        ["s0 zwevend (ISO 76)",    r["s0_zwevend"],       "-"],
        ["benutting n vast",       r["benutting_n_vast"], "-"],
        ["benutting n zwevend",    r["benutting_n_zwevend"], "-"],
    ], columns=["grootheid", "waarde", "eenheid"])


def plot_lagerlevensduur(controle: Dict, cfg: ConfiguratieLagers) -> None:
    r    = controle["resultaten"]
    doel = controle["doel_h"]
    labels   = ["L10h vast", "L10h zwevend"]
    waarden  = [r["L10h_vast"], r["L10h_zwevend"]]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    colors = ['#2ca02c' if w >= doel else '#d62728' for w in waarden]
    axes[0].bar(labels, waarden, color=colors)
    axes[0].axhline(doel, color='k', linestyle='--', label=f"Doel: {doel:.0f} h")
    axes[0].set_ylabel("L10 levensduur [h]")
    axes[0].set_title("Lagerlevensduur vs. eis")
    axes[0].legend()
    axes[0].grid(True, axis='y', alpha=0.4)

    # Equivalente belasting vs capaciteit
    labels2   = ["P_eq vast", "P_eq zwevend", "Cdyn vast", "Cdyn zwevend"]
    waarden2  = [r["P_eq_vast_N"], r["P_eq_zwevend_N"],
                 cfg_vast.C_dyn_N, cfg_zwevend.C_dyn_N]
    axes[1].bar(labels2, waarden2, color=['#1f77b4','#ff7f0e','#1f77b4','#ff7f0e'],
                alpha=[1,1,0.4,0.4])
    axes[1].set_ylabel("Kracht [N]")
    axes[1].set_title("P_eq vs. Cdyn")
    axes[1].grid(True, axis='y', alpha=0.4)

    plt.tight_layout()
    plt.show()

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# HOOFDBEREKENING
# ─────────────────────────────────────────────────────────────────────────────
iface_exp   = resultaten_interface["export_voor_volgende_modules"]
spindel_exp = resultaten_spindel["export_voor_volgende_modules"]

n_rpm_run   = float(spindel_exp.get("n_bedrijf_rpm",   840.0))
spoed_m_run = float(spindel_exp.get("spoed_spindel_m", 0.020))

cfg_lagers  = ConfiguratieLagers(
    lagerspanning_m = float(spindel_exp.get("m1_schacht_totaal_m", 2.478)),
)
cfg_vast    = ConfiguratieVastLager()
cfg_zwevend = ConfiguratieZwevendLager()

lagerkrachten   = bepaal_lagerkrachten(iface_exp, spindel_exp, cfg_lagers)
controle_lagers = verifieer_lagers(lagerkrachten, cfg_lagers, cfg_vast, cfg_zwevend, spindel_exp)
overzicht_lagers = maak_overzicht_lagers(lagerkrachten, controle_lagers, cfg_vast, cfg_zwevend)

In [12]:
print("\n=== LAGERKRACHTEN ===")
print(pd.Series(lagerkrachten).to_string())
toon_dataframe("overzicht lagers", overzicht_lagers)
toon_dataframe("pass/fail lagers", controle_lagers["controles"])
print("\n=== CONTROLERESULTATEN LAGERS ===")
print(pd.Series(controle_lagers["resultaten"]).to_string())

n_ok = int(controle_lagers["controles"]["geslaagd"].sum())
n_tot = len(controle_lagers["controles"])
print(f"\n>>> LAGERS VERIFICATIE: {n_ok}/{n_tot} checks geslaagd")


=== LAGERKRACHTEN ===
Fz_N               1179.300000
My_Nm               527.000000
F_axiaal_N         1832.000000
R_gewicht_N         589.650000
R_moment_N          212.671509
F_rad_vast_N        802.321509
F_rad_zwevend_N     376.978491

=== OVERZICHT LAGERS ===
           grootheid                         waarde eenheid
          vast lager HIWIN WBK30DF (2× 7206 BE, DF)       -
       zwevend lager              HIWIN BK30 (6206)       -
F_axiaal (vastlager)                         1832.0       N
      F_radiaal vast                     802.321509       N
   F_radiaal zwevend                     376.978491       N
           P_eq vast                    2038.461464       N
        P_eq zwevend                          390.0       N
           Cdyn vast                        24570.0       N
        Cdyn zwevend                        19500.0       N
           L10h vast                   15814.204798       h
        L10h zwevend                 1128884.264979       h
           Doe

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# EXPORT
# ─────────────────────────────────────────────────────────────────────────────
resultaten_lagers = {
    "configuratie_lagers":   asdict(cfg_lagers),
    "configuratie_vast":     asdict(cfg_vast),
    "configuratie_zwevend":  asdict(cfg_zwevend),
    "lagerkrachten":         lagerkrachten.copy(),
    "controle_lagers": {
        "resultaten": controle_lagers["resultaten"].copy(),
        "controles":  controle_lagers["controles"].copy(),
    },
    "overzicht_lagers": overzicht_lagers.copy(),
    "export_voor_volgende_modules": {
        "L10h_vast":        controle_lagers["resultaten"]["L10h_vast"],
        "L10h_zwevend":     controle_lagers["resultaten"]["L10h_zwevend"],
        "L10km_vast":       controle_lagers["resultaten"]["L10km_vast"],
        "L10km_zwevend":    controle_lagers["resultaten"]["L10km_zwevend"],
        "s0_vast":          controle_lagers["resultaten"]["s0_vast"],
        "s0_zwevend":       controle_lagers["resultaten"]["s0_zwevend"],
        "P_eq_vast_N":      controle_lagers["resultaten"]["P_eq_vast_N"],
        "P_eq_zwevend_N":   controle_lagers["resultaten"]["P_eq_zwevend_N"],
        "marge_L10h_vast":  controle_lagers["resultaten"]["marge_L10h_vast"],
        "naam_vast":        cfg_vast.naam,
        "naam_zwevend":     cfg_zwevend.naam,
    },
}

print("\n=== EXPORT VOOR VOLGENDE MODULES ===")
print(pd.Series(resultaten_lagers["export_voor_volgende_modules"]).to_string())


=== EXPORT VOOR VOLGENDE MODULES ===
L10h_vast                            15814.204798
L10h_zwevend                       1128884.264979
L10km_vast                           15940.718436
L10km_zwevend                      1137915.339099
s0_vast                                 12.950944
s0_zwevend                              28.717949
P_eq_vast_N                           2038.461464
P_eq_zwevend_N                              390.0
marge_L10h_vast                            2.7484
naam_vast          HIWIN WBK30DF (2× 7206 BE, DF)
naam_zwevend                    HIWIN BK30 (6206)
